# 06 — EDA readiness

Issue the final go/no-go gate for downstream exploratory analysis. This notebook does not perform EDA, signal preprocessing, feature engineering, or scientific-value changes.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in {"notebooks", "data"}:
    PROJECT_ROOT = PROJECT_ROOT.parent
required_directories = ("data", "models", "notebooks", "results")
if not all((PROJECT_ROOT / name).is_dir() for name in required_directories):
    raise RuntimeError("Run this notebook from bci_cleaning/ or its notebooks/ directory")

RAW_PATH = PROJECT_ROOT / "data" / "raw"
CLEANED_PATH = PROJECT_ROOT / "data" / "processed"
REPORTS_PATH = PROJECT_ROOT / "results" / "reports"
RULES_PATH = PROJECT_ROOT / "data" / "documentation_rules.json"
EXPECTED_RAW = PROJECT_ROOT.parent / "BCI Database"

if not RAW_PATH.is_symlink() or RAW_PATH.resolve(strict=True) != EXPECTED_RAW.resolve(strict=True):
    raise RuntimeError("data/raw must remain a relative symlink to the sibling BCI Database directory")

os.chdir(PROJECT_ROOT)
support_path = str(PROJECT_ROOT / "support")
if support_path not in sys.path:
    sys.path.insert(0, support_path)

In [2]:
from eda_readiness import main

_exit_code = main([
    "--raw", str(RAW_PATH),
    "--cleaned", str(CLEANED_PATH),
    "--reports", str(REPORTS_PATH),
    "--rules", str(RULES_PATH),
])
if _exit_code != 0:
    raise RuntimeError(f"EDA-readiness gate returned nonzero status {_exit_code}")

EDA readiness: PASS


In [3]:
import csv

report_path = REPORTS_PATH / "eda_readiness.csv"
with report_path.open("r", encoding="utf-8", newline="") as handle:
    report_rows = list(csv.DictReader(handle))

{
    "report": str(report_path.relative_to(PROJECT_ROOT)),
    "checks": len(report_rows),
    "passed": sum(row["status"] == "pass" for row in report_rows),
}

{'report': 'results/reports/eda_readiness.csv', 'checks': 10, 'passed': 10}